In [7]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

X, y = make_classification(n_samples = 1000, weights = [0.9, 0.1],
                           n_informative = 5, random_state = 42)
model = LogisticRegression()

#Basic 5-Fold cross-validation
scores = cross_val_score(model, X, y, cv=5)
print(f"cross-val score: {scores.mean():.3f} ± {scores.std():.3f}")

#Stratified 5 Fold cv
strat_cv = StratifiedKFold(n_splits = 5, shuffle = True, random_state=42)
strat_scores = cross_val_score(model, X, y, cv=strat_cv)
print(f"Stratified cross-val score: {scores.mean():.3f} ± {scores.std():.3f}")

# See individual fold scores
print(f"Per-fold scores: {strat_scores.round(3)}")


cross-val score: 0.894 ± 0.016
Stratified cross-val score: 0.894 ± 0.016
Per-fold scores: [0.915 0.895 0.9   0.885 0.9  ]


Avoiding Data Leakage with Pipeline

In [12]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import make_classification

X, y = make_classification(n_samples = 1000, n_features = 10, random_state = 42)

# WRONG: Scaler sees ALL data including test set
# scaler = StandardScaler()
# X_scaled = scaler.fit_transform(X)  # Leakage!
# scores = cross_val_score(LogisticRegression(), X_scaled, y, cv=5)

#Correct way - Pipeline ensures us that scaler fits only on training folds
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', LogisticRegression(max_iter=1000))
])

scores = cross_val_score(pipe, X, y, cv=5)
print(f"Pipeline scores: {scores.mean():.3f} ± {scores.std()}")


Pipeline scores: 0.856 ± 0.01593737745050924


Time Series cross validation

In [20]:
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Ridge
import numpy as np

# Simulated time series (features and target)
np.random.seed(42)
n_samples = 200
X = np.random.randn(n_samples, 5)  # 5 features
y = np.cumsum(np.random.randn(n_samples))

tcsv = TimeSeriesSplit()
model = Ridge()
scores = []

for fold, (train_idx, test_idx) in enumerate(tcsv.split(X)):
  X_train, X_test = X[train_idx], X[test_idx]
  y_train, y_test = y[train_idx], y[test_idx]

  model.fit(X_train, y_train)
  score = model.score(X_test, y_test)
  scores.append(score)
  print(f"Fold {fold + 1} -> Train[:{train_idx[-1]+1}], Test[{test_idx[0]}: {test_idx[-1]+1}] -> R^2 = {score:.3f}")

  print(f"Time Series CV: {np.mean(scores):.3f} +- {np.std(scores):.3f}")

Fold 1 -> Train[:35], Test[35: 68] -> R^2 = -8.324
Time Series CV: -8.324 +- 0.000
Fold 2 -> Train[:68], Test[68: 101] -> R^2 = -2.283
Time Series CV: -5.304 +- 3.020
Fold 3 -> Train[:101], Test[101: 134] -> R^2 = -5.173
Time Series CV: -5.260 +- 2.467
Fold 4 -> Train[:134], Test[134: 167] -> R^2 = -50.653
Time Series CV: -16.608 +- 19.771
Fold 5 -> Train[:167], Test[167: 200] -> R^2 = -18.414
Time Series CV: -16.969 +- 17.699
